# eval_run — chấm RQ1 (BPC test + Top-1/Top-3 + KSR) cho các checkpoint (ĐG, issue #24)
1. Settings: GPU T4 · Internet On · Environment **Pin to original environment (2026-11-19)**.
2. Add Input: Dataset `mamba-btl-data`, `mamba-btl-wheels` **và output (version) của notebook huấn luyện** từng lượt cần chấm (có `checkpoints/<run_name>/ckpt_latest.pt`).
3. Sửa danh sách `RUNS` ở cell 1.
4. Bấm **Save Version → Save & Run All** (chạy nền). Kết quả nằm ở tab **Output** của version, thư mục `logs/`: `<run>_eval.json` và `<run>_suggest.jsonl` — tải về rồi PR vào `logs/` của repo.
   Đừng chạy ở phiên tương tác rồi Stop session: file trong `/kaggle/working` của phiên tương tác mất khi tắt phiên.

Thời gian trên T4 (fp16): Mamba L256 ≈ 8 phút/lượt, Transformer L256 ≈ 5 phút/lượt (2.000 vị trí + 50 đoạn KSR); L dài hơn lâu hơn do prefill dài hơn. BPC toàn bộ test chạy đầu tiên, chỉ ≈ 1 phút.

Đã chấm Top-k/KSR rồi, chỉ cần bổ sung BPC test: đặt `EXTRA = "--skip_topk --skip_ksr"`. Cell 2 chép `logs/<run>_eval.json` đã có trong repo sang `/kaggle/working/logs`, nên khóa `bpc` được gộp vào cùng file, giữ nguyên `topk`, `ksr`.

In [ ]:
# 1. CHỈ SỬA Ở ĐÂY
RUNS = ["mamba_L256_s0", "transformer_L256_s0"]   # các lượt cần chấm (đã Add Input output chứa checkpoint)
EXTRA = ""          # vd "--limit 200 --skip_ksr --skip_bpc" để chạy thử; "--skip_topk --skip_ksr" chỉ BPC; "--l_ctx 2048" ngoại suy
BRANCH = "main"        # nhánh code đánh giá ("dg/suggest" nếu PR #41 chưa merge)

In [ ]:
# 2. Clone repo + cài mamba-ssm từ wheel
%cd /kaggle/working
!rm -rf mamba-vi-keyboard && git clone -q -b {BRANCH} https://github.com/daisyngocha/mamba-vi-keyboard.git
%cd /kaggle/working/mamba-vi-keyboard
!git log --oneline -1
from utils import kaggle_input
wheel_dir = kaggle_input("mamba-btl-wheels")
wheels = sorted(str(p) for p in wheel_dir.rglob("*.whl")) if wheel_dir else []
assert wheels or not any(r.startswith("mamba") for r in RUNS), "Chưa Add Input Dataset mamba-btl-wheels"
if wheels:
    !pip install -q --no-deps {" ".join(wheels)}
!pip install -q pyyaml
assert kaggle_input("mamba-btl-data"), "Chưa Add Input Dataset mamba-btl-data"
# _eval.json đã có trong repo -> chép sang thư mục output để kết quả mới (vd BPC) gộp vào, không mất topk/ksr
import shutil
from pathlib import Path
Path("/kaggle/working/logs").mkdir(exist_ok=True)
for r in RUNS:
    src, dst = Path(f"logs/{r}_eval.json"), Path(f"/kaggle/working/logs/{r}_eval.json")
    if src.exists() and not dst.exists():
        shutil.copy(src, dst); print("chép sẵn", src)

In [ ]:
# 3. Kiểm tra: GPU, mamba-ssm, đủ checkpoint cho mọi lượt
import torch, sys
print(torch.__version__, torch.version.cuda, sys.version.split()[0], torch.cuda.get_device_name())
from evaluate.common import find_checkpoint
missing = []
for r in RUNS:
    ck = find_checkpoint(r)
    print(f"{r:24s} -> {ck}")
    if ck is None:
        missing.append(r)
assert not missing, f"Chưa Add Input output notebook chứa checkpoint của: {missing}"

In [ ]:
# 4. Chấm từng lượt: BPC test -> Top-k -> KSR (tiến độ in mỗi 20 batch BPC / 200 vị trí / 10 đoạn KSR)
for r in RUNS:
    !python -m evaluate.ksr --run {r} {EXTRA}

In [ ]:
# 5. Bảng tổng hợp
import json, pandas as pd
from pathlib import Path
rows = []
for r in RUNS:
    p = Path(f"/kaggle/working/logs/{r}_eval.json")
    if not p.exists():
        continue
    e = json.loads(p.read_text(encoding="utf-8"))
    t, k = e.get("topk", {}), e.get("ksr", {})
    rows.append({"run": r, "bpc_test": e.get("bpc", {}).get("bpc"), "top1": t.get("all", {}).get("top1"), "top3": t.get("all", {}).get("top3"),
                 "top3_a": t.get("a", {}).get("top3"), "top3_b": t.get("b", {}).get("top3"),
                 "top3_nocase": t.get("all", {}).get("top3_nocase"), "ksr": k.get("ksr"),
                 "ms_per_item": t.get("ms_per_item"), "dtype": t.get("dtype")})
df = pd.DataFrame(rows)
print(df.round(4).to_string(index=False))
!ls -la /kaggle/working/logs